# Natural Disaster Chat Application

**A RAG + MCP chatbot for natural disaster data analysis and knowledge retrieval.**

This notebook composes two modules:
- **RAG Pipeline**: Hybrid retrieval (Dense + BM25 + RRF + Reranking) over PDF knowledge base
- **MCP Server**: CSV query tool over EM-DAT disaster data (1970–2021)
- **LangGraph Agent**: Routes queries to the appropriate module based on intent

## Sections
1. Setup & Configuration
2. Data Ingestion
3. Retrieval Demo
4. MCP Server Demo
5. Agent Chat
6. Evaluation
7. Visualizations
8. Architecture Diagrams

In [ ]:
# --- Section 1: Setup & Configuration ---
import os
import sys
import warnings
warnings.filterwarnings("ignore")

# Ensure src/ is on the path
sys.path.insert(0, os.path.abspath("."))

from dotenv import load_dotenv
load_dotenv()

from src.config import (
    CSV_PATH, PDF_DIR, CHROMA_PERSIST_DIR, OUTPUT_DIR,
    HF_EMBEDDING_MODEL, LLM_MODEL, LLM_TEMPERATURE,
)

print(f"CSV path: {CSV_PATH} (exists: {CSV_PATH.exists()})")
print(f"PDF dir: {PDF_DIR}")
print(f"ChromaDB dir: {CHROMA_PERSIST_DIR}")
print(f"Embedding model: {HF_EMBEDDING_MODEL}")
print(f"LLM model: {LLM_MODEL}")

## 2. Data Ingestion _(feeds RAG pipeline)_

Load CSV disaster data and optional PDF documents, then chunk and embed into ChromaDB.

> **Data flow:** CSV rows + PDFs → LangChain Documents → Parent chunks (2048 chars) → Child chunks (512 chars) → ChromaDB embeddings.
> These chunks are used exclusively by the **RAG path** (hybrid retrieval). The MCP path reads the raw CSV directly via Pandas.

In [ ]:
# --- Section 2: Data Ingestion (RAG pipeline preparation) ---
# This step converts raw data into embeddings for the RAG retrieval path.
# The MCP path does NOT use these chunks — it queries the CSV directly via Pandas.
from src.ingestion.loaders import load_pdfs, load_csv_as_docs
from src.ingestion.chunking import build_parent_child_chunks

# Load CSV as documents (each row → one LangChain Document for embedding)
csv_docs = load_csv_as_docs(CSV_PATH)
print(f"CSV documents loaded: {len(csv_docs)}")
print(f"Sample: {csv_docs[0].page_content[:120]}")

# Load PDFs (if available)
pdf_docs = load_pdfs(PDF_DIR)
print(f"\nPDF documents loaded: {len(pdf_docs)}")

# Hierarchical chunking: Parent (2048 chars) for context, Child (512 chars) for retrieval
all_docs = csv_docs + pdf_docs
parent_chunks, child_chunks = build_parent_child_chunks(all_docs)
print(f"\nParent chunks: {len(parent_chunks)}")
print(f"Child chunks: {len(child_chunks)}")

In [ ]:
# --- Embed child chunks into ChromaDB (RAG vector store) ---
# ChromaDB is the vector store backing the RAG pipeline's dense retrieval leg.
from src.retrieval.vectorstore import create_vectorstore, load_vectorstore, get_embeddings
from pathlib import Path

embeddings = get_embeddings(prefer_hf=True)

if Path(CHROMA_PERSIST_DIR).exists() and any(Path(CHROMA_PERSIST_DIR).iterdir()):
    print("Loading existing vectorstore...")
    vectorstore = load_vectorstore(CHROMA_PERSIST_DIR, embeddings)
else:
    print(f"Creating vectorstore with {len(child_chunks)} child chunks...")
    vectorstore = create_vectorstore(child_chunks, CHROMA_PERSIST_DIR, embeddings)

print(f"Vectorstore ready (collection count: {vectorstore._collection.count()})")

## 3. Retrieval Demo _(RAG path only)_

Demonstrate the hybrid retriever (Dense + BM25 + RRF) and Cohere reranker.

> **Data flow (RAG):** User query → Dense search (ChromaDB, top-20) + BM25 sparse search (top-20) → Reciprocal Rank Fusion (top-15) → Cohere Reranker (top-5) → retrieved chunks.
> This is the retrieval mechanism used when the agent classifies intent as `knowledge_base`.

In [ ]:
# --- Section 3: Retrieval Demo (RAG path — hybrid search + reranking) ---
# This demonstrates the RAG retrieval pipeline in isolation, before the agent is involved.
# Flow: query → Dense (ChromaDB) + Sparse (BM25) → RRF fusion → Cohere reranker
from src.retrieval.hybrid import HybridRetriever
from src.retrieval.reranker import rerank

# Build hybrid retriever (used by the agent's "knowledge_base" and "mixed" routes)
hybrid_retriever = HybridRetriever(vectorstore, child_chunks)

# Demo query — exercises the RAG path directly
query = "What are the deadliest earthquakes?"
results = hybrid_retriever.retrieve(query)
print(f"Hybrid retrieval for: '{query}'")
print(f"Results: {len(results)} documents\n")

for i, doc in enumerate(results[:5]):
    print(f"  [{i+1}] {doc.page_content[:100]}...")
    print(f"      Metadata: {doc.metadata.get('country', 'N/A')} | {doc.metadata.get('year', 'N/A')}\n")

# Apply Cohere reranking (final stage of RAG retrieval)
reranked = rerank(query, results)
print(f"\nAfter reranking: {len(reranked)} documents")
for i, doc in enumerate(reranked[:3]):
    print(f"  [{i+1}] {doc.page_content[:100]}...")

## 4. MCP Server Demo _(MCP path only)_

Query the EM-DAT disaster CSV through the MCP server tool — no vector store involved.

> **Data flow (MCP):** Filter parameters → `query_natural_disasters()` tool → Pandas DataFrame filtering on raw CSV → McpResponse envelope `{ok, data, error, meta}`.
> This is the path used when the agent classifies intent as `disaster_data`.

In [ ]:
# --- Section 4: MCP Server Demo (MCP path — structured CSV queries, no RAG) ---
# This calls the FastMCP tool directly. The MCP path reads the raw CSV via Pandas,
# applies filters (country, year, type, min_deaths), and returns structured rows.
# No embeddings or vector store are involved.
from src.mcp_server.server import query_natural_disasters
import json

# Query 1: Earthquakes in Japan (MCP: country + disaster_type filter)
result = query_natural_disasters(country="Japan", disaster_type="Earthquake")
print(f"Japan Earthquakes: {result['data']['total_matching']} records")
print(f"Timing: {result['meta']['timing_ms']:.1f}ms\n")

# Query 2: Deadliest disasters in 2010 (MCP: year filter + sort)
result = query_natural_disasters(year=2010, sort_by="Total Deaths", limit=5)
print("Top 5 deadliest disasters in 2010:")
for row in result["data"]["rows"][:5]:
    print(f"  {row['Country']} - {row['Disaster Type']}: {row.get('Total Deaths', 'N/A')} deaths")

# Query 3: Year range with minimum deaths (MCP: year_range + min_deaths filter)
print("\nDisasters 2000-2010 with 50,000+ deaths:")
result = query_natural_disasters(year_start=2000, year_end=2010, min_deaths=50000, sort_by="Total Deaths")
for row in result["data"]["rows"]:
    print(f"  {row['Year']} {row['Country']} - {row['Disaster Type']}: {int(row['Total Deaths']):,} deaths")

## 5. Agent Chat _(LangGraph routes to RAG, MCP, or both)_

The LangGraph agent classifies each query's intent and routes it accordingly:

| Intent | Route | What runs |
|--------|-------|-----------|
| `knowledge_base` | **RAG only** | Hybrid retriever → reranker → LLM generation |
| `disaster_data` | **MCP only** | LLM extracts filters → Pandas CSV query → LLM generation |
| `mixed` | **RAG + MCP** | Both paths execute, contexts merged, then LLM generation |
| `general` | **Direct LLM** | No retrieval — LLM answers directly |

The three demo queries below each exercise a different route.

In [ ]:
# --- Section 5: Agent Chat ---
from langchain_openai import ChatOpenAI
from src.agent.graph import build_agent

# Initialize LLM
llm = ChatOpenAI(model=LLM_MODEL, temperature=LLM_TEMPERATURE)

# Build the agent with retriever and reranker
agent = build_agent(llm, retriever=hybrid_retriever, reranker_fn=rerank)
print("Agent built successfully!")

# Chat function
def chat(question: str, history: list = None):
    """Send a question to the agent and display the response."""
    if history is None:
        history = []
    result = agent.invoke({
        "question": question,
        "intent": "",
        "context": "",
        "answer": "",
        "chat_history": history,
        "sources": [],
    })
    print(f"Intent: {result['intent']}")
    print(f"\n{result['answer']}")
    return result

In [ ]:
# --- Demo: disaster_data intent → MCP path only ---
# The agent classifies this as a structured data question and routes to the CSV query tool.
# Flow: question → classify("disaster_data") → LLM extracts filters → Pandas query → LLM answer
result1 = chat("How many earthquakes occurred in Japan between 2000 and 2020? list by year")

In [ ]:
# --- Demo: knowledge_base intent → RAG path only ---
# The agent classifies this as a conceptual question and routes to the hybrid retriever.
# Flow: question → classify("knowledge_base") → Dense+BM25 → RRF → Reranker → LLM answer
result2 = chat("What causes earthquakes and how do early warning systems work?")

In [ ]:
# --- Demo: mixed intent → RAG + MCP paths combined ---
# The agent classifies this as needing both data and knowledge, so it runs BOTH paths.
# Flow: question → classify("mixed") → [RAG retrieval] + [CSV query] → merge contexts → LLM answer
result3 = chat("Why was the 2010 Haiti earthquake so deadly compared to other earthquakes?")

## 6. Evaluation

Evaluate using the golden dataset (50 Q&A pairs) and compare retrieval strategies.

> **Evaluation scope:**
> - **MCP evaluation:** CSV query correctness — do the Pandas filters return expected results for 20 data questions?
> - **RAG evaluation:** Hit rate@5 — do the top-5 retrieved chunks contain relevant keywords for 20 knowledge questions?
> - **Strategy comparison (RAG):** Dense-only vs BM25-only vs Hybrid vs Hybrid+Rerank — measured by hit rate@5.

In [ ]:
# --- Section 6a: MCP path evaluation — CSV query correctness ---
# Tests whether the MCP tool returns correct results for the 20 "disaster_data" golden questions.
from src.evaluation.evaluator import (
    load_golden_dataset, evaluate_csv_query_correctness, run_full_evaluation
)
from src.evaluation.dashboard import render_metrics_table, render_csv_correctness_chart

# Load golden dataset
golden = load_golden_dataset()
print(f"Golden dataset: {len(golden)} items")

# CSV query correctness (MCP path evaluation)
csv_eval = evaluate_csv_query_correctness(golden)
print(f"\nCSV Query Correctness: {csv_eval['correct']}/{csv_eval['total']} = {csv_eval['accuracy']:.1%}")

# Display correctness chart
fig = render_csv_correctness_chart(csv_eval["details"])
fig.show()

In [ ]:
# --- Section 6b: RAG path evaluation — 4-strategy comparison ---
# Compares retrieval quality across 4 RAG strategies using "knowledge_base" golden questions.
# Strategies: Dense-only, BM25-only, Hybrid (Dense+BM25+RRF), Hybrid+Rerank
from src.evaluation.strategy_comparison import compare_strategies, render_comparison_report

strategy_results = compare_strategies(vectorstore, child_chunks, reranker_fn=rerank)

print("Strategy Comparison — Hit Rate@5:")
for name, vals in strategy_results.items():
    print(f"  {name}: {vals['hit_rate_at_5']:.4f}")

# Render comparison charts
table_fig, chart_fig = render_comparison_report(strategy_results)
table_fig.show()
chart_fig.show()

## 7. Visualizations _(data extracted via MCP/Pandas path)_

Interactive charts from the EM-DAT disaster dataset. These read the CSV directly via Pandas
(same data source as the MCP path), not through the RAG vector store.

In [ ]:
# --- Section 7: Visualizations ---
from src.visualization.charts import create_all_charts

charts = create_all_charts()
for name, fig in charts.items():
    print(f"Rendering: {name}")
    fig.show()

In [ ]:
# --- Network Diagram ---
from src.visualization.diagrams import country_disaster_network

network_fig = country_disaster_network()
network_fig.show()

In [ ]:
# --- Generate HTML Report ---
from src.visualization.report import generate_html_report

report_path = generate_html_report()
print(f"HTML report saved to: {report_path}")
print(f"File size: {report_path.stat().st_size / 1024:.0f} KB")

## 8. Architecture Diagrams

### System Architecture

```mermaid
graph TD
    A[User] --> B[Jupyter Notebook Chat UI]
    B --> C[LangGraph Agent - ReAct Routing]
    C --> D{Intent Classification}
    D -->|disaster_data| E[MCP Server]
    D -->|knowledge_base| F[RAG Pipeline]
    D -->|mixed| G[Both Paths]
    D -->|general| H[Direct LLM Response]

    E --> I[CSV Query Tool - Pandas]
    I --> J[(EM-DAT CSV 1970-2021)]

    F --> K[Hybrid Retriever]
    K --> L[Dense Search - ChromaDB]
    K --> M[Sparse Search - BM25]
    L --> N[Reciprocal Rank Fusion]
    M --> N
    N --> O[Cohere Reranker]

    G --> E
    G --> F

    O --> P[LLM Generation - GPT-4o-mini]
    I --> P
    H --> P
    P --> Q[Response with Citations]
    Q --> B

    style A fill:#3498db,color:#fff
    style C fill:#2ecc71,color:#fff
    style E fill:#e74c3c,color:#fff
    style F fill:#9b59b6,color:#fff
    style J fill:#f39c12,color:#fff
    style L fill:#1abc9c,color:#fff
    style P fill:#e67e22,color:#fff
```

### RAG Pipeline Detail

```mermaid
graph LR
    A[PDF Documents] --> B[PyPDF Loader]
    C[CSV Data] --> D[Row-to-Doc Converter]
    B --> E[Parent Chunker 2048 chars]
    D --> E
    E --> F[Child Chunker 512 chars]
    F --> G[HuggingFace Embeddings BGE-M3]
    G --> H[(ChromaDB Vector Store)]

    I[User Query] --> J[Dense Search top-20]
    I --> K[BM25 Sparse Search top-20]
    H --> J
    J --> L[RRF Fusion top-15]
    K --> L
    L --> M[Cohere Reranker top-5]
    M --> N[LLM Generation]
    N --> O[Answer + Citations]

    style H fill:#3498db,color:#fff
    style L fill:#2ecc71,color:#fff
    style M fill:#e74c3c,color:#fff
    style N fill:#e67e22,color:#fff
```

## 9. Streamlit Chat UI

Launch the interactive chat interface powered by the same LangGraph agent.
Once running, open [http://localhost:8501](http://localhost:8501) in your browser.

In [ ]:
# 🚀 Launch the Streamlit chat app (opens in browser at http://localhost:8501)
import subprocess, sys, os

app_dir = os.path.dirname(os.path.abspath("streamlit_app.py"))
_streamlit_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "streamlit_app.py", "--server.port", "8501"],
    cwd=app_dir,
)
print(f"App started → http://localhost:8501  (PID {_streamlit_proc.pid})")

In [ ]:
# 🛑 Stop the Streamlit app
try:
    _streamlit_proc.terminate()
    _streamlit_proc.wait(timeout=5)
    print(f"App stopped (PID {_streamlit_proc.pid})")
except NameError:
    import subprocess
    subprocess.run(["pkill", "-f", "streamlit run"], check=False)
    print("App stopped")